# 1: Align  DNA and RNA FISH images

1) Get global coorrdinates for DNA and RNA FISH beads.
2) Get local and glolbal alignment for images.
3) Apply alignment to images.

For more details about required in and output please look at the individual notebooks used in the pipeline.

In [ ]:
import os
import queue
import pandas as pd
from pathlib import Path
from datetime import datetime
import papermill as pm
from multiprocessing import Pool
import concurrent.futures


def run_notebook(parameters,notebook_to_run,parameters_common={},out= '/dev/null'):
    
    # change to directory where the notebook is (resolve relative imports)
    os.chdir(Path(notebook_to_run).absolute().parent)
    
    # run notebook
    for parameters_spec in parameters_list:
        parameters = {**parameters_common, **parameters_spec}

        pm.execute_notebook(
            notebook_to_run,
            out,
            parameters=parameters)

# 1) Get global coordinates

In [ ]:
parameters_list = [
#     {"in_path": "/data/agl_data/NanoFISH/Gabi/GS204_RNA-DNA-FISH_sequential_test/20240307_RNAFISH/"},
#     {"in_path": "/data/agl_data/NanoFISH/Gabi/GS204_RNA-DNA-FISH_sequential_test/20240310_DNAFISH/"}
#     {"in_path": "/data/agl_data/NanoFISH/Gabi/GS485_Dppa3_all_mESC_RNA-DNA/20240725_RNAFISH/"},
#     {"in_path": "/data/agl_data/NanoFISH/Gabi/GS485_Dppa3_all_mESC_RNA-DNA/20240727_DNAFISH/"},
    # {"in_path": "/data/agl_data/NanoFISH/Gabi/GS487_Dppa3_all_mESC_RNA-DNA/20240802_RNAFISH/"},
    # {"in_path": "/data/agl_data/NanoFISH/Gabi/GS487_Dppa3_all_mESC_RNA-DNA/20240803_DNAFISH/"}
    # {"in_path": "/data/agl_data/NanoFISH/Gabi/GS593_Dppa3_all_mESC_DNA-RNA/20240930_RNA/"},
    # {"in_path": "/data/agl_data/NanoFISH/Gabi/GS593_Dppa3_all_mESC_DNA-RNA/20241002_DNAFISH/"}
    {"in_path": "/data/agl_data/NanoFISH/Gabi/GS602_Dppa3_DNA_RNA/20241026_RNAFISH/"},
    {"in_path": "/data/agl_data/NanoFISH/Gabi/GS602_Dppa3_DNA_RNA/20241030_DNAFISH/"}
]

### common parameters
parameters_common = {
    "raw_subpath" : "raw", # subpath where raw h5/msr images are saved
    "spots_subpath": "detections_beads/merge.csv", # single or multiple (*) spot files
    "out_subpath" : "detections_beads/", #w here to save results
    
    "coordinate_column_names" : ['z', 'y', 'x'], # naming of xyz coordinates
    "global_coordinate_column_names" : ['z_global_um', 'y_global_um', 'x_global_um'], # how to call global coordinate columns
    "image_file_column_name" : 'img', # name of img file column
    
    "use_h5_for_metadata" : True, # use h5 (faster) or msr? 
}

notebook_to_run = "/home/stumberger/image-analyis-recipes/alignment/sted/get_global_coordinates_sted.ipynb"

run_notebook(parameters_list,notebook_to_run,parameters_common)

# 2) Get alignment for images

In [ ]:
parameters_list = [
    {
        "base_path_target": "/data/agl_data/NanoFISH/Gabi/GS204_RNA-DNA-FISH_sequential_test/20240310_DNAFISH/",
        "base_path_moving": "/data/agl_data/NanoFISH/Gabi/GS204_RNA-DNA-FISH_sequential_test/20240307_RNAFISH/"
    }
]


parameters_common = {
    "coordinates_path_target" : "detections_beads/merge_global_coords.csv",
    "coordinates_path_moving" : "detections_beads/merge_global_coords.csv",
    
    "save_subdir" : "alignment_parameters",
    
    "coordinate_column_names" : ['z_global_um', 'y_global_um', 'x_global_um'],
    "image_file_column_name" : "img",
    
    # in order of complexity:
    # euclidean: move & rotate, similarity: + scale, affine: + shear
    "transform_type" : "similarity",
    
    # when doing local alignment, radius around center of moving image to consider (in um)
    "match_radius" : 30.0
}
    
notebook_to_run = "/home/stumberger/image-analyis-recipes/alignment/sted/find_transformations_sted.ipynb"

run_notebook(parameters_list,notebook_to_run,parameters_common)

# 3) Align bead images

In [ ]:
parameters_list = [
    # {
#         "base_path_target": "/data/agl_data/NanoFISH/Gabi/GS204_RNA-DNA-FISH_sequential_test/20240310_DNAFISH/",
#         "base_path_moving": "/data/agl_data/NanoFISH/Gabi/GS204_RNA-DNA-FISH_sequential_test/20240307_RNAFISH/"
    # },
    # {
    #     "base_path_target": "/data/agl_data/NanoFISH/Gabi/GS487_Dppa3_all_mESC_RNA-DNA/20240803_DNAFISH/",
    #     "base_path_moving": "/data/agl_data/NanoFISH/Gabi/GS487_Dppa3_all_mESC_RNA-DNA/20240802_RNAFISH/"
    # }

        {
        "base_path_target": "/data/agl_data/NanoFISH/Gabi/GS485_Dppa3_all_mESC_RNA-DNA/20240727_DNAFISH/",
        "base_path_moving": "/data/agl_data/NanoFISH/Gabi/GS485_Dppa3_all_mESC_RNA-DNA/20240725_RNAFISH/"
    }
]


parameters_common = {
    "alignment_params_file_moving" : 'alignment_parameters/alignment_parameters_intensity_based.json',
    "file_exclude_pattern_target" : "sted",
    "file_include_pattern_target" : None,
    "file_exclude_pattern_moving" : None,
    "file_include_pattern_moving" : None,

    "channels_to_include_target" : [2, ],
    "channels_to_include_moving" : [1, ],
    
    # "channels_to_include_target" : [0, ],
    # "channels_to_include_moving" : [0, ],
    
    # whether to fuse multiple moving images
    # if False, will only transform the one with the highest overlap, ignoring other moving tiles at the border of target image
    "fuse_multiple_moving" : True,
    
    "out_subdir" : 'aligned_beads_correlation',
    # "out_subdir" : 'aligned_beads_global',
    
    # whether to save projections or not plus folder to save them to (will be subdir of out_subdir)
    "save_projections": True,
    "projections_subdir": 'vis'
}
    
notebook_to_run = "/home/stumberger/image-analyis-recipes/alignment/sted/apply_alignment_to_images.ipynb"

run_notebook(parameters_list,notebook_to_run,parameters_common)

# 4) Align FISH images

In [ ]:
parameters_list = [
    # {
    #     "base_path_target": "/data/agl_data/NanoFISH/Gabi/GS487_Dppa3_all_mESC_RNA-DNA/20240803_DNAFISH/",
    #     "base_path_moving": "/data/agl_data/NanoFISH/Gabi/GS487_Dppa3_all_mESC_RNA-DNA/20240802_RNAFISH/"
    # }

    {
        "base_path_target": "/data/agl_data/NanoFISH/Gabi/GS602_Dppa3_DNA_RNA/20241030_DNAFISH/",
        "base_path_moving": "/data/agl_data/NanoFISH/Gabi/GS602_Dppa3_DNA_RNA/20241026_RNAFISH//"
    }
]


parameters_common = {
    "alignment_params_file_moving" : 'alignment_parameters/alignment_parameters_local.json',
    "file_exclude_pattern_target" : None,
    "file_include_pattern_target" : "sted",
    "file_exclude_pattern_moving" : None,
    "file_include_pattern_moving" : None,
    
    # for spots
    "channels_to_include_target" : [0,1, ],
    "channels_to_include_moving" : [0, ],
    
    # for beads
    # "channels_to_include_target" : [2, ],
    # "channels_to_include_moving" : [1, ],
    
    # whether to fuse multiple moving images
    # if False, will only transform the one with the highest overlap, ignoring other moving tiles at the border of target image
    "fuse_multiple_moving" : True,
    
    "out_subdir" : 'aligned_sted',
    
    # whether to save projections or not plus folder to save them to (will be subdir of out_subdir)
    "save_projections": True,
    "projections_subdir": 'vis'
}
    
notebook_to_run = "/home/stumberger/image-analyis-recipes/alignment/sted/apply_alignment_to_images.ipynb"

run_notebook(parameters_list,notebook_to_run,parameters_common)

In [ ]:


parameters_list = [
#     {"in_path": "/data/agl_data/NanoFISH/Gabi/GS204_RNA-DNA-FISH_sequential_test/20240307_RNAFISH/"},
    {"in_path": "/data/agl_data/NanoFISH/Gabi/GS204_RNA-DNA-FISH_sequential_test/20240310_DNAFISH/"}
]

### common parameters
parameters_common = {
    "raw_subpath" : "raw", # subpath where raw h5/msr images are saved
    "spots_subpath": "detections_sted/merge.csv", # single or multiple (*) spot files
    "out_subpath" : "detections_sted/", #w here to save results
    
    "coordinate_column_names" : ['z', 'y', 'x'], # naming of xyz coordinates
    "global_coordinate_column_names" : ['z_global_um', 'y_global_um', 'x_global_um'], # how to call global coordinate columns
    "image_file_column_name" : 'img', # name of img file column
    
    "use_h5_for_metadata" : True, # use h5 (faster) or msr? 
}

notebook_to_run = "/home/stumberger/image-analyis-recipes/alignment/sted/get_global_coordinates_sted.ipynb"

run_notebook(parameters_list,notebook_to_run,parameters_common)